# SpliceTransformer Splice Site Prediction

![SpliceTransformer Splice Site Prediction](https://proto-bio.github.io/proto-assets/images/tool/splice_transformer/hero.png)

This notebook demonstrates `run_splice_transformer`, which predicts tissue-specific splice site locations from raw DNA sequence. Given a target sequence flanked by genomic context, the model returns per-position probabilities for three splice categories (neither, acceptor, donor) along with 15 tissue-specific usage channels, making it useful for variant interpretation, synthetic intron design, and analysis of tissue-specific alternative splicing.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("splice_transformer")
display_overview("splice_transformer")
display_docs_section("splice_transformer", "Background")

## Available tools

In [ ]:
display_available_tools("splice_transformer")

### `run_splice_transformer`

Given a target DNA sequence flanked by left and right genomic context, `run_splice_transformer` uses a transformer architecture trained on RNA-seq data from 15 human tissues to predict per-position splice site probabilities. The output tensor has 18 channels: three splice type channels (neither, acceptor, donor) that sum to 1.0 at every position, followed by 15 tissue-specific usage channels covering adipose, blood, brain, heart, liver, and other tissues. This structure makes it straightforward to identify canonical splice sites and to compare their predicted usage across tissues.

In [ ]:
import numpy as np

from proto_tools import (
    SPLICE_TISSUE_CHANNEL_INDEX,
    SpliceTransformerType,
    SpliceTransformerConfig,
    SpliceTransformerInput,
    run_splice_transformer,
)

In [ ]:
# Display input docs
display_api_reference("splice_transformer", "input", "run_splice_transformer")

from pathlib import Path
from Bio import SeqIO

# Load human beta-globin (HBB) genomic sequence (NG_059281.1)
# HBB is a small, well-characterized gene with 3 exons and 2 introns
fasta_path = Path("./hbb_genomic.fasta")
record = SeqIO.read(fasta_path, "fasta")
full_seq = str(record.seq)
print(f"Gene: HBB (Homo sapiens hemoglobin subunit beta)")
print(f"Accession: {record.id}")
print(f"Total length: {len(full_seq):,}bp")

# Targets must be exactly 1000 bp; left/right contexts exactly 4000 bp.
target_start = 4700
target_end = 5700
target_seq = full_seq[target_start:target_end]
left_ctx = full_seq[target_start - 4000:target_start]
right_ctx = full_seq[target_end:target_end + 4000]

print(f"\nTarget: positions {target_start}-{target_end} ({len(target_seq)}bp)")
print(f"Context: {len(left_ctx)}bp left, {len(right_ctx)}bp right")

inputs = SpliceTransformerInput(
    target_seqs=[target_seq],
    left_contexts=[left_ctx],
    right_contexts=[right_ctx],
)

In [ ]:
# Display config docs
display_api_reference("splice_transformer", "config", "run_splice_transformer")

config = SpliceTransformerConfig(
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=True,
)

In [ ]:
# Run the prediction
result = run_splice_transformer(inputs, config)

In [ ]:
# Display output docs
display_api_reference("splice_transformer", "output", "run_splice_transformer")

pred = np.array(result.prediction)
print(f"Prediction shape: {pred.shape}")
print(f"  batch={pred.shape[0]}, target_length={pred.shape[1]}, channels={pred.shape[2]}")

## Interpreting Predictions

The prediction tensor has 18 channels. Channels 0 through 2 correspond to splice type probabilities: "neither" (not a splice site), "acceptor", and "donor". Channels 3 through 17 provide tissue-specific splice usage predictions for 15 human tissues. The splice type channels sum to 1.0 at each position, representing a probability distribution over the three categories.

In [ ]:
pred = np.array(result.prediction[0])  # First (only) sequence

# Splice type probabilities
neither_probs = pred[:, SpliceTransformerType.NEITHER.value]
acceptor_probs = pred[:, SpliceTransformerType.ACCEPTOR.value]
donor_probs = pred[:, SpliceTransformerType.DONOR.value]

print(f"Max acceptor probability: {acceptor_probs.max():.4f} at position {acceptor_probs.argmax()}")
print(f"Max donor probability: {donor_probs.max():.4f} at position {donor_probs.argmax()}")
print(f"Mean 'neither' probability: {neither_probs.mean():.4f}")

## Finding High-Confidence Splice Sites

A threshold on acceptor and donor probabilities identifies positions where the model is confident a splice site exists. A threshold of 0.5 typically captures the strongest splice sites while filtering noise.

In [ ]:
threshold = 0.5

acceptor_sites = np.where(acceptor_probs > threshold)[0]
donor_sites = np.where(donor_probs > threshold)[0]

print(f"High-confidence acceptor sites (>{threshold}): {len(acceptor_sites)}")
if len(acceptor_sites) > 0:
    print(f"  Positions: {acceptor_sites[:10]}..." if len(acceptor_sites) > 10 else f"  Positions: {acceptor_sites}")

print(f"High-confidence donor sites (>{threshold}): {len(donor_sites)}")
if len(donor_sites) > 0:
    print(f"  Positions: {donor_sites[:10]}..." if len(donor_sites) > 10 else f"  Positions: {donor_sites}")

## Tissue-Specific Analysis

Channels 3 through 17 provide splice usage predictions for 15 human tissues including brain, heart, liver, and muscle. Comparing these channels reveals tissue-specific alternative splicing, where the same splice site is used at different rates in different tissues. This is biologically important because alternative splicing is a major mechanism for generating protein diversity from a single gene.

In [ ]:
pred = np.array(result.prediction[0])
for tissue_name, channel_idx in SPLICE_TISSUE_CHANNEL_INDEX.items():
    if channel_idx is None:  # AVERAGE is handled separately via channel slice 3:
        continue
    tissue_pred = pred[:, channel_idx]
    print(f"{tissue_name:20s}: mean={tissue_pred.mean():.4f}, max={tissue_pred.max():.4f}")

### `run_splice_transformer_gradient`

The gradient tool turns these predictions into differentiable quantities. Each term names an output channel — a splice class (`NEITHER`/`ACCEPTOR`/`DONOR`) or one of the fifteen tissues — and a span of target positions, and comes back with that channel's mean probability and its own raw gradient; a direction and a weight are goals the caller applies afterwards.

Only the **target region** is relaxed. A candidate bundles the two fixed 4,000 bp flanking contexts with a `(1000, 4)` relaxed target matrix, so the gradient covers the target positions alone while both flanks still shape the prediction. That matches how the model is used: the flanks are genomic context you are not designing.

Here the term covers the strongest acceptor site found above.

In [ ]:
from proto_tools import (
    SpliceTransformerGradientCandidate, SpliceTransformerGradientConfig,
    SpliceTransformerGradientInput, SpliceTransformerProbabilityTerm,
    run_splice_transformer_gradient,
)

display_api_reference("splice-transformer-gradient", "input", "run_splice_transformer_gradient")

# Relax the same target region: a sharpened one-hot of the real sequence, so the starting
# distribution favours the wild-type base at each position without saturating.
grad_sharpness = 2.0
grad_logits = [
    [grad_sharpness if base == observed else 0.0 for base in "ACGT"]
    for observed in target_seq.upper()
]
grad_candidate = SpliceTransformerGradientCandidate(
    left_context=left_ctx,
    right_context=right_ctx,
    logits=grad_logits,
)
print(f"contexts: {len(left_ctx)} + {len(right_ctx)} bp | relaxed target: {len(grad_logits)} bp")


In [ ]:
display_api_reference("splice-transformer-gradient", "config", "run_splice_transformer_gradient")

# Target an 11 bp window centred on the strongest acceptor position. Term positions are
# 1-indexed and relative to the target region, the same frame `acceptor_probs` indexes.
grad_peak = int(acceptor_probs.argmax())
grad_config = SpliceTransformerGradientConfig(
    gradient_terms=[
        SpliceTransformerProbabilityTerm(
            channel="ACCEPTOR",
            position_start=max(1, grad_peak + 1 - 5),
            position_end=min(len(grad_logits), grad_peak + 1 + 5),
        )
    ],
)
print(f"strongest acceptor at target position {grad_peak} (p={acceptor_probs[grad_peak]:.4f})")

grad_output = run_splice_transformer_gradient(
    SpliceTransformerGradientInput(inputs=grad_candidate), grad_config
)


In [ ]:
display_api_reference("splice-transformer-gradient", "output", "run_splice_transformer_gradient")

grad_result = grad_output.results[0]
grad_term = grad_result.terms[0]
print(f"Target length:      {grad_result.metrics['target_length']}")
print(f"Gradient shape:     ({len(grad_term.gradient)}, {len(grad_term.gradient[0])})")
print(f"  {grad_term.term.channel} positions "
      f"{grad_term.term.position_start}-{grad_term.term.position_end}: "
      f"mean probability {grad_term.term_value:.4f}")

# The derivative at the acceptor base itself, over A, C, G, T.
print(f"Gradient at the acceptor base: {[round(v, 6) for v in grad_term.gradient[grad_peak]]}")


## Export Results

Predictions can be exported to NumPy format for downstream analysis, including integration with variant effect prediction pipelines or visualization in genome browsers.

In [ ]:
result.export("splice_transformer_predictions", export_path="./example_output", file_format="npy")
print("Predictions exported to ./example_output/splice_transformer_predictions.npy")